In [ ]:
import ast
import pandas as pd
import numpy as np
import ipynbname
from Testing.DRTLO_GD_R2 import *
from Functions.AutoCloud import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
import optuna
from optuna.samplers import RandomSampler
from optuna.exceptions import TrialPruned

RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = HI['PC1'].values

def df_ParamsTable(names):
    if len(names) == 13:
        FileName = ipynbname.name()
        study_dir = f'Optimization/1{FileName}/multi/'
        out_path = f'Optimization/1{FileName}/multi/Optimization.csv'
        study_dir = os.path.normpath(study_dir)
        out_path = os.path.normpath(out_path)
        os.makedirs(study_dir, exist_ok=True)
    elif len(names) == 12:
        FileName = ipynbname.name()
        study_dir = f'Optimization/1{FileName}/mono/'
        out_path = f'Optimization/1{FileName}/mono/Optimization.csv'
        study_dir = os.path.normpath(study_dir)
        out_path = os.path.normpath(out_path)
        os.makedirs(study_dir, exist_ok=True)

    return study_dir, out_path

def to_nearest_pow10(val):
    return 10 ** np.round(np.log10(val))

def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(multivariate=True,group=True,n_startup_trials=1000)
    elif mode == 'random':
        sampler=RandomSampler()
    return sampler

class EarlyStoppingCallback:
    def __init__(self, patience: int):
        self.patience = patience
        self.best_score = float('inf')
        self.trials_without_improvement = 0

    def __call__(self, study: optuna.study.Study, trial: optuna.trial.FrozenTrial):
        if trial.state != optuna.trial.TrialState.COMPLETE:
            return

        # USA trial.values (lista com [wape_RUL, wape_HI])
        # Somamos os valores para avaliar a melhoria geral
        current_score = sum(trial.values)

        if current_score < self.best_score:
            self.best_score = current_score
            self.trials_without_improvement = 0
        else:
            self.trials_without_improvement += 1

        if self.trials_without_improvement >= self.patience:
            print(f"O estudo parou! O erro não diminui há {self.patience} iterações.")
            study.stop()


In [ ]:
params =  [2.0, 5, '[58, 11, 64, 31, 32]', 15, 5, 0.0754386941522406,6.688979613494848e-06, 0.0029919547064531, 10.168345750070529,'ahead', 'tanh']

m,nI,nR,nO,mO,N1,N2,N3,τ,mode,act = params
X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
nI = len(Y[0])
if isinstance(nR, str): nR = ast.literal_eval(nR)

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
               tau=τ,rho=0.001,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
for j,_ in enumerate(X[:]):
    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
    teda.Adapt(Y[j],Z[j])

teda.c = np.append(teda.c,teda.gm)  
PlotDSI_3D_PLT(teda)


In [ ]:
print(xxx)

In [ ]:
names = ['MAPE_RUL*MAPE_HI', 'm', 'nI', 'nR', 'nO', 'mO', 'N1', 'N2', 'N3','TAU','past/ahead','activation']
df2 = []
n_study = 1
time_break = 2700
n_trials = 11
n_patience = int(n_trials*0.2)
nMax = 20

study_dir, out_path = df_ParamsTable(names)

for i in range(n_study):
    print('iteration:',i+1)
    def objective(trial):
        m = trial.suggest_float('m', 1.75, 4.25,step=0.25)
        nI = trial.suggest_int('nI', 2, nMax) 
        n_layers = trial.suggest_int('n_layers', 2, 5)
        nR = [trial.suggest_int(f'nR_layer_{l}', 1, 60) for l in range(n_layers)]
        nO = trial.suggest_int('nO', 1, nMax) 
        mO = trial.suggest_int('mO', 0, nMax) 
        N1 = trial.suggest_int('N1', 1, 9)
        N2 = trial.suggest_int('N2', 1, 9)
        N3 = trial.suggest_int('N3', 1, 9)
        τ = trial.suggest_float('τ', 1, 25,log=True)
        mode = trial.suggest_int('mode', 1,1)  
        act = trial.suggest_int('act', 0,1)  
        n1,n2,n3 = (1/(10**N1)),(1/(10**N2)),(1/(10**N3))
        if mode == 'past':
            if nO > nI: raise TrialPruned()
        elif mode == 'ahead':
            if nI> nMax or nO> nMax: raise TrialPruned()
            elif mO > nI: raise TrialPruned()
            
        X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
        nI = len(Y[0])

        
        teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO+mO,ηS=[n1,n2,n3],mode=mode,act=act,
                    tau=τ,rho=0.0,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
        for j,_ in enumerate(X[:]):
            teda.run(X[j])
            teda.RUL_Prediction(Y[j],mode='single',lim=len(sig)-nI+1)
            teda.Adapt(Y[j],Z[j])

            if j%35 == 0:
                trial.report(teda.wape_RUL+teda.wape_HI, j)
            if trial.should_prune():
                raise optuna.TrialPruned()
            '''if j == 50:
                if teda.wape_RUL > 0.5: raise TrialPruned()
                if teda.wape_HI  > 0.5: raise TrialPruned()

            if j%35 == 0 and len(teda.wape_HI_hist==j):
                if teda.wape_RUL > 0.6: raise TrialPruned()
                elif teda.wape_HI  > 0.6: raise TrialPruned()
                elif np.sum(teda.wape_HI_hist[:j])/j > 0.6: raise TrialPruned()'''

            if trial.should_prune():
                raise optuna.TrialPruned()

        return teda.wape_RUL+teda.wape_HI

    pruner=optuna.pruners.HyperbandPruner()
    early_stopping = EarlyStoppingCallback(patience=n_patience)
    study = optuna.create_study(direction='minimize',pruner=pruner,sampler=SelSampler(mode='tpe'))
    study.optimize(objective, n_trials=n_trials, timeout=time_break, callbacks=[early_stopping])

    trials = study.best_trials
    trials_errors = [list(trial.values) for trial in trials]
    trials_params = [list(trial.params.values()) for trial in trials]

    for trial in study.best_trials:
        p = trial.params
        nR_list = [p[f'nR_layer_{l}'] for l in range(p['n_layers'])]
        row = [trial.values[0],p['m'],p['nI'],str(nR_list),p['nO'],p['mO'],p['N1'],p['N2'],p['N3'],p['τ'],p['mode'],p['act']]
        df2.append(row)

df2 = pd.DataFrame(df2, columns=names)

if os.path.isfile(out_path) and out_path.startswith(study_dir):
    df1 = pd.read_csv(out_path)
    df_stdy = pd.concat([df1, df2], ignore_index=True)

else: df_stdy = df2

df_stdy.to_csv(out_path, index=False)
opt_path = os.path.join(study_dir,f'opt_{len(os.listdir(study_dir))-1}.csv')
df2.to_csv(opt_path, index=False)


In [11]:
names = ['MAPE_RUL','MAPE_HI', 'm', 'nI', 'nR', 'nO', 'mO', 'N1', 'N2', 'N3','TAU','past/ahead','activation']
df2 = []
n_study = 1
time_break = 300
n_trials = 11
n_patience = int(n_trials*0.1)
nMax = 20

study_dir, out_path = df_ParamsTable(names)

for i in range(n_study):
    print('iteration:',i+1)
    def objective(trial):
        m = trial.suggest_float('m', 1.75, 4.25,step=0.25)
        nI = trial.suggest_int('nI', 2, nMax) 
        n_layers = trial.suggest_int('n_layers', 1, 5)
        nR = [trial.suggest_int(f'nR_layer_{l}', 1, 60) for l in range(n_layers)]
        nO = trial.suggest_int('nO', 1, nMax) 
        mO = trial.suggest_int('mO', 0, nMax) 
        N1 = trial.suggest_int('N1', 1, 9)
        N2 = trial.suggest_int('N2', 1, 9)
        N3 = trial.suggest_int('N3', 1, 9)
        τ = trial.suggest_float('τ', 1, 25,log=True)
        mode = trial.suggest_int('mode', 1,1)  
        act = trial.suggest_int('act', 0,1)  
        n1,n2,n3 = (1/(10**N1)),(1/(10**N2)),(1/(10**N3))
        if mode == 'past':
            if nO > nI: raise TrialPruned()
        elif mode == 'ahead':
            if nI> nMax or nO> nMax: raise TrialPruned()
            elif mO > nI: raise TrialPruned()
            
        X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
        nI = len(Y[0])

        teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO+mO,ηS=[n1,n2,n3],mode=mode,act=act,
                    tau=τ,rho=0.0,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
        for j,_ in enumerate(X[:]):
            teda.run(X[j])
            teda.RUL_Prediction(Y[j],mode='single',lim=len(sig)-nI+1)
            teda.Adapt(Y[j],Z[j])

            if j%35 == 0 and len(teda.wape_HI_hist==j):
                if teda.wape_RUL > 0.6: raise TrialPruned()
                elif teda.wape_HI  > 0.6: raise TrialPruned()
                elif np.sum(teda.wape_HI_hist[:j])/j > 0.6: raise TrialPruned()
            if len(teda.wape_RUL_hist==j):
                if np.sum(teda.wape_RUL_hist[:j])/j > 0.6: raise TrialPruned()

        return teda.wape_RUL,teda.wape_HI

    early_stopping = EarlyStoppingCallback(patience=n_patience)
    study = optuna.create_study(directions=['minimize','minimize'],sampler=SelSampler(mode='auto'))
    study.optimize(objective, n_trials=n_trials, timeout=time_break, callbacks=[early_stopping])

    trials = study.best_trials
    trials_errors = [list(trial.values) for trial in trials]
    trials_params = [list(trial.params.values()) for trial in trials]

    for trial in study.best_trials:
        p = trial.params
        nR_list = [p[f'nR_layer_{l}'] for l in range(p['n_layers'])]
        row = [trial.values[0],trial.values[1],p['m'],p['nI'],str(nR_list),p['nO'],p['mO'],p['N1'],p['N2'],p['N3'],p['τ'],p['mode'],p['act']]
        df2.append(row)

df2 = pd.DataFrame(df2, columns=names)

if os.path.isfile(out_path) and out_path.startswith(study_dir):
    df1 = pd.read_csv(out_path)
    df_stdy = pd.concat([df1, df2], ignore_index=True)

else: df_stdy = df2

df_stdy.to_csv(out_path, index=False)
opt_path = os.path.join(study_dir,f'opt_{len(os.listdir(study_dir))-1}.csv')
df2.to_csv(opt_path, index=False)


[I 2026-08-12 12:55:11,508] A new study created in memory with name: no-name-e5c2038d-f5fe-4479-b310-b024ec8ac2fe
[I 2026-08-12 12:55:11,513] Trial 0 pruned. 
[I 2026-08-12 12:55:11,517] Trial 1 pruned. 
[I 2026-08-12 12:55:11,519] Trial 2 pruned. 
[I 2026-08-12 12:55:11,522] Trial 3 pruned. 
[I 2026-08-12 12:55:11,525] Trial 4 pruned. 
[I 2026-08-12 12:55:11,527] Trial 5 pruned. 
[I 2026-08-12 12:55:11,530] Trial 6 pruned. 
[I 2026-08-12 12:55:11,533] Trial 7 pruned. 
[I 2026-08-12 12:55:11,536] Trial 8 pruned. 
[I 2026-08-12 12:55:11,540] Trial 9 pruned. 
[I 2026-08-12 12:55:11,546] Trial 10 pruned. 


iteration: 1


In [ ]:
out_path = 'Optimization\\eDRTLO_GD_R02_mono\\Optimization.csv'
df_params = pd.read_csv(out_path)
#df_params = pd.read_csv(opt_path)
df_params = df_params[(df_params.iloc[:,0] <= 0.07)]
params_list = df_params.values[:,-11:]
df_params

In [ ]:
for i,params in enumerate(params_list):
    m,nI,nR,nO,mO,N1,N2,N3,τ,mode,act = params
    X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
    nI=len(Y[0])
    if isinstance(nR, str): nR = ast.literal_eval(nR)

    teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
                tau=τ,rho=0.001,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
    for j,_ in enumerate(X[:]):
        teda.run(X[j])
        teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
        teda.Adapt(Y[j],Z[j])

    teda.c = np.append(teda.c,teda.gm)  
    #PlotSeriesPLY(ySeries=[teda.wape_HI_hist,teda.wape_RUL_hist])
    PlotDSI_3D_PLT(teda)

In [ ]:
m,nI,nR,nO,mO,N1,N2,N3,τ,mode,act = params_list[0]
X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
nI = len(Y[0])
if isinstance(nR, str): nR = ast.literal_eval(nR)
n1,n2,n3 = (1/(10**N1)),(1/(10**N2)),(1/(10**N3))

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO+mO,ηS=[n1,n2,n3],mode=mode,act=act,
               tau=τ,rho=0.001,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
for j,_ in enumerate(X[:]):
    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
    teda.Adapt(Y[j],Z[j])
teda.c = np.append(teda.c,teda.gm)  
PlotDSI_3D_PLT(teda)
